# 2MU7 Heavy-Atom to Residue Geometry Roundtrip

This notebook demonstrates the PHEAT real-PDB roundtrip using the committed `2MU7` fixture: heavy atoms are loaded from PDB, converted to the residue-geometry representation, reconstructed as heavy atoms, scored before and after reconstruction, compared with all-heavy, backbone, and C-alpha RMSD plus radius of gyration, aligned with Kabsch superposition, and visualized with Mol* through `ipymolstar` and MolViewSpec.


In [ ]:
from dataclasses import replace
from pathlib import Path
import sys
ROOT = Path.cwd().resolve()
for candidate in [ROOT, *ROOT.parents]:
    if (candidate / "pyproject.toml").exists():
        ROOT = candidate
        break
else:
    raise FileNotFoundError("Could not find repository root containing pyproject.toml")

src_path = str(ROOT / "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

from IPython.display import Markdown, display  # noqa: E402

from pheat import kabsch_align, kabsch_rmsd  # noqa: E402
from pheat.metrics import radius_of_gyration_delta, structure_radius_of_gyration  # noqa: E402
from pheat.models import HeavyAtomStructure  # noqa: E402
from pheat.pdbio import load_pdb, structure_to_pdb_string  # noqa: E402
from pheat.scoring import score_structure  # noqa: E402
from pheat.residue_geometry import structure_from_residue_geometry, structure_to_residue_geometry  # noqa: E402

def atom_key(atom):
    return (
        atom.chain_id or "",
        int(atom.resseq),
        atom.icode or "",
        atom.resname.strip().upper(),
        atom.name.strip().upper(),
    )

def atoms_by_key(structure):
    return {atom_key(atom): atom for atom in structure.atoms}

def markdown_table(headers, rows):
    header = "| " + " | ".join(headers) + " |"
    separator = "| " + " | ".join("---" for _ in headers) + " |"
    body = ["| " + " | ".join(str(value) for value in row) + " |" for row in rows]
    return "\n".join([header, separator, *body])

def fmt(value):
    if isinstance(value, float):
        return f"{value:.15g}"
    return value


## Roundtrip Through Residue Geometry

The default reconstruction omits terminal `OXT`, matching the source fixture's atom identity for this example.

In [ ]:
original_pdb = ROOT / "tests" / "fixtures" / "2mu7.pdb"

original = load_pdb(original_pdb)
residue_geometry = structure_to_residue_geometry(original)
reconstructed = structure_from_residue_geometry(residue_geometry)

original_atoms = atoms_by_key(original)
reconstructed_atoms = atoms_by_key(reconstructed)
common_keys = sorted(set(original_atoms) & set(reconstructed_atoms))
missing_after = sorted(set(original_atoms) - set(reconstructed_atoms))
extra_after = sorted(set(reconstructed_atoms) - set(original_atoms))

display(Markdown(
    f"Original heavy atoms: `{len(original.atoms)}`  \n"
    f"Reconstructed heavy atoms: `{len(reconstructed.atoms)}`  \n"
    f"Matched atom keys: `{len(common_keys)}`  \n"
    f"Missing after reconstruction: `{len(missing_after)}`  \n"
    f"Extra after reconstruction: `{len(extra_after)}`  \n"
    f"Original has OXT: `{any(atom.name.strip().upper() == 'OXT' for atom in original.atoms)}`  \n"
    f"Reconstructed has OXT: `{any(atom.name.strip().upper() == 'OXT' for atom in reconstructed.atoms)}`"
))


## Kabsch Alignment And RMSD

`kabsch_align` returns reconstructed coordinates optimally superposed onto the original coordinates. `kabsch_rmsd` can either perform that alignment internally or reuse the aligned coordinates.

In [ ]:
original_coords = [original_atoms[key].coord for key in common_keys]
reconstructed_coords = [reconstructed_atoms[key].coord for key in common_keys]
aligned_coords = kabsch_align(original_coords, reconstructed_coords)

backbone_keys = [key for key in common_keys if key[4] in {"N", "CA", "C", "O"}]
backbone_original_coords = [original_atoms[key].coord for key in backbone_keys]
backbone_reconstructed_coords = [reconstructed_atoms[key].coord for key in backbone_keys]
backbone_aligned_coords = kabsch_align(backbone_original_coords, backbone_reconstructed_coords)

ca_keys = [key for key in common_keys if key[4] == "CA"]
ca_original_coords = [original_atoms[key].coord for key in ca_keys]
ca_reconstructed_coords = [reconstructed_atoms[key].coord for key in ca_keys]
ca_aligned_coords = kabsch_align(ca_original_coords, ca_reconstructed_coords)

rmsd_rows = [
    [
        "all common heavy atoms",
        len(common_keys),
        fmt(kabsch_rmsd(original_coords, reconstructed_coords, aligned_target=aligned_coords)),
    ],
    [
        "common backbone atoms N/CA/C/O",
        len(backbone_keys),
        fmt(kabsch_rmsd(
            backbone_original_coords,
            backbone_reconstructed_coords,
            aligned_target=backbone_aligned_coords,
        )),
    ],
    [
        "common C-alpha atoms",
        len(ca_keys),
        fmt(kabsch_rmsd(ca_original_coords, ca_reconstructed_coords, aligned_target=ca_aligned_coords)),
    ],
]
display(Markdown(markdown_table(["Atom set", "Matched atoms", "Kabsch RMSD (A)"], rmsd_rows)))

aligned_by_key = dict(zip(common_keys, aligned_coords))
aligned_atoms = []
for atom in reconstructed.atoms:
    coord = aligned_by_key.get(atom_key(atom))
    if coord is None:
        aligned_atoms.append(atom)
    else:
        aligned_atoms.append(replace(atom, x=float(coord[0]), y=float(coord[1]), z=float(coord[2])))

aligned_reconstructed = HeavyAtomStructure(
    atoms=aligned_atoms,
    name=f"{reconstructed.name}:kabsch_aligned",
    metadata={**reconstructed.metadata, "alignment": "kabsch_to_original_2mu7"},
)


## Radius Of Gyration

Radius of gyration summarizes the structure's spatial spread around its geometric or mass-weighted center. It is invariant to the rigid-body Kabsch alignment used for RMSD.

In [ ]:
original_rg = structure_radius_of_gyration(original)
reconstructed_rg = structure_radius_of_gyration(reconstructed)
rg_delta = radius_of_gyration_delta(original_rg, reconstructed_rg)

rg_rows = []
for key, label in [("unweighted", "unweighted"), ("mass_weighted", "mass-weighted")]:
    rg_rows.append([
        label,
        fmt(original_rg["values"].get(key)),
        fmt(reconstructed_rg["values"].get(key)),
        fmt(rg_delta["values"].get(key)),
        original_rg["units"],
    ])

display(Markdown(markdown_table(
    ["Mode", "Original heavy", "After residue-geometry reconstruction", "Delta", "Units"],
    rg_rows,
)))


## Energy Comparisons

The built-in scoring models are deterministic heavy-atom approximations with arbitrary units. The large shifts in `generic` and `heavy-mm` mainly reflect close contacts introduced by idealized residue-geometry reconstruction, not physical experimental energies.

In [ ]:
models = ["generic", "pheat-dfire", "pheat-goap", "heavy-mm"]
energy_rows = []
for model in models:
    before = score_structure(original, model=model)
    after = score_structure(reconstructed, model=model)
    energy_rows.append([
        f"`{model}`",
        fmt(before.total),
        fmt(after.total),
        fmt(after.total - before.total),
        after.units,
    ])

display(Markdown(markdown_table(
    ["Model", "Original heavy", "After residue-geometry reconstruction", "Delta", "Units"],
    energy_rows,
)))


## Optional OpenMM Prepared Scoring

This cell only reports OpenMM/AMBER values when OpenMM can parameterize the structures. Failures are shown as notes instead of stopping the notebook.

In [ ]:
openmm_rows = []
for label, structure in [
    ("original heavy", original),
    ("after residue-geometry reconstruction", reconstructed),
]:
    try:
        result = score_structure(structure, model="openmm-prepared")
        openmm_rows.append([label, fmt(result.total), result.units, "ok"])
    except Exception as exc:
        openmm_rows.append([label, "not available", "", str(exc)])

display(Markdown(markdown_table(["Structure", "OpenMM total", "Units", "Status"], openmm_rows)))


## Mol* Alignment Visualization

The reconstructed structure is first Kabsch-aligned to the original heavy atoms. The viewer receives a single local PDB payload embedded as a data URI, uses Mol* default coloring, and assigns original atoms to display chain `A` and reconstructed atoms to display chain `B`.


In [ ]:
from importlib import metadata

diagnostic_rows = [["Python executable", f"`{sys.executable}`"]]
try:
    import ipymolstar
    import molviewspec
except Exception as exc:
    diagnostic_rows.append(["Mol* notebook imports", f"failed: `{exc}`"])
else:
    diagnostic_rows.append(["ipymolstar __version__", getattr(ipymolstar, "__version__", "unknown")])
    diagnostic_rows.append(["molviewspec __version__", getattr(molviewspec, "__version__", "unknown")])

for package in ["anywidget", "ipywidgets", "jupyterlab_widgets"]:
    try:
        version = metadata.version(package)
    except metadata.PackageNotFoundError:
        version = "not installed"
    diagnostic_rows.append([package, version])

display(Markdown(markdown_table(["Component", "Value"], diagnostic_rows)))


In [ ]:
try:
    import base64
    from ipymolstar import MolViewSpec
    from molviewspec import create_builder
except Exception as exc:
    display(Markdown(
        "Mol* notebook viewer support is not available in this environment. "
        "Install the notebook extra or use the Miniforge environment from `environment.yml`.  \n"
        f"Error: `{exc}`"
    ))
else:
    original_display_atoms = [
        replace(atom, chain_id="A", serial=None, occupancy=1.0, bfactor=0.0)
        for atom in original.atoms
    ]
    reconstructed_display_atoms = [
        replace(atom, chain_id="B", serial=None, occupancy=1.0, bfactor=0.0)
        for atom in aligned_reconstructed.atoms
    ]
    combined_display = HeavyAtomStructure(
        atoms=[*original_display_atoms, *reconstructed_display_atoms],
        name="2mu7-original-reconstructed-aligned",
    )
    combined_display_pdb = structure_to_pdb_string(combined_display)
    encoded_pdb = base64.b64encode(combined_display_pdb.encode("utf-8")).decode("ascii")
    pdb_data_uri = f"data:chemical/x-pdb;base64,{encoded_pdb}"

    builder = create_builder().canvas(background_color="white")
    structure = builder.download(url=pdb_data_uri).parse(format="pdb").model_structure()
    structure.component(selector={"auth_asym_id": "A"}).representation(type="cartoon")
    structure.component(selector={"auth_asym_id": "B"}).representation(type="cartoon")
    state = builder.get_state(title="2MU7 original and reconstructed heavy atoms")
    try:
        msvj_data = state.model_dump_json(exclude_none=True)
    except AttributeError:
        msvj_data = state.json(exclude_none=True)

    display(MolViewSpec(msvj_data=msvj_data, height="560px"))
